# Controlled model comparison (issue #45)

Two distinct approved approaches, a regularized linear model (E01) and gradient-boosted trees (E02), are trained on the frozen `OCT-SPLIT-001` training partition and compared against the required training-mean constant and the training-median comparator.

**Sources this notebook builds on (all by teammates, read before writing):**
- Split and baselines: `docs/team/october/Gate 1/split_design_decision.md`, `notebooks/final-deliverables/October/Gate 2/` (Ella Zou, Liam Stapley)
- Preprocessing and experiment plan (E01/E02 definitions, rare-level threshold 132, categorical mask): `docs/team/october/Gate 1/preprocessing_plan.md`, `experiment_plan.md` (Mia Chavez, branch `42-preprocessing-and-experiment-plan`)
- Cross-check only: Jonathan Deng's `notebooks/jonathan-deng/october_modeling_outputs/` (branch `jonathan`). His numbers are compared in the last section; nothing here is copied from them.

**Rules kept:** fit only on train; tune only with the 5 recorded `cv_fold` folds (pipeline refit per fold); validation is a limited comparison check; test rows are dropped at load time; MAE is in original `loss` units; seed 42.

In [ ]:
import hashlib, json, platform, time, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEED = 42
RARE_MIN_FREQUENCY = 132  # 0.1% of train, from preprocessing_plan.md
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data" / "allstate_claims_data.csv"
MANIFEST = ROOT / "notebooks" / "final-deliverables" / "October" / "Gate 1" / "partition_manifest.csv"
OUT = ROOT / "notebooks" / "junaid-pathan" / "october" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
EXPECTED_SOURCE_SHA = "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03"
EXPECTED_MANIFEST_SHA = "de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b"
WORKFLOW_RUN = "OCT-JP-MODEL-COMPARISON-001"

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

assert sha256(DATA) == EXPECTED_SOURCE_SHA, "source file differs from the frozen version"
assert sha256(MANIFEST) == EXPECTED_MANIFEST_SHA, "manifest differs from OCT-SPLIT-001"
versions = {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
            "pandas": pd.__version__, "numpy": np.__version__}
print(versions)
if sklearn.__version__ != "1.6.1":
    print("NOTE: preprocessing_plan.md pins scikit-learn 1.6.1; this run used", sklearn.__version__)

## 1. Load the frozen split
Merge the manifest on `id`, then drop the protected test rows immediately.

In [ ]:
source = pd.read_csv(DATA)
manifest = pd.read_csv(MANIFEST)
df = source.merge(manifest, on="id", how="inner", validate="one_to_one")
df = df[df["partition"] != "test"].reset_index(drop=True)   # test is never loaded for modeling

train = df[df["partition"] == "train"].reset_index(drop=True)
validation = df[df["partition"] == "validation"].reset_index(drop=True)
CAT = [f"cat{i}" for i in range(1, 117)]
CONT = [f"cont{i}" for i in range(1, 15)]
FEATURES = CAT + CONT

assert (len(train), len(validation)) == (131_822, 28_248)
assert set(df["partition"]) == {"train", "validation"}
assert train["cv_fold"].isin(range(5)).all() and validation["cv_fold"].isna().all()
assert df[FEATURES].isna().sum().sum() == 0          # missing-value guard from the plan
assert "loss" not in FEATURES and "id" not in FEATURES
print(train["cv_fold"].value_counts().sort_index().to_dict())
pd.DataFrame({"partition": ["train", "validation"], "claims": [len(train), len(validation)]})

## 2. Shared evaluation design
One function scores every model, so baselines and candidates use the same claims and the same original-unit MAE. Cross-validation uses the manifest's `cv_fold` column and refits the whole pipeline inside each fold, so vocabularies and scalers never see the held-out fold.

In [ ]:
def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))   # original loss units

def cross_validate(make_pipeline, frame=train, target_transform=None):
    """5 recorded folds; pipeline refit per fold. Returns fold MAEs, mean, sd, seconds."""
    start = time.perf_counter()
    fold_maes = []
    for fold in range(5):
        fit_rows, hold_rows = frame[frame["cv_fold"] != fold], frame[frame["cv_fold"] == fold]
        model = make_pipeline()
        y = fit_rows["loss"] if target_transform is None else target_transform[0](fit_rows["loss"])
        model.fit(fit_rows[FEATURES], y)
        pred = model.predict(hold_rows[FEATURES])
        if target_transform is not None:
            pred = target_transform[1](pred)
        fold_maes.append(mae(hold_rows["loss"], pred))
    return {"fold_maes": fold_maes, "cv_mae_mean": float(np.mean(fold_maes)),
            "cv_mae_sd": float(np.std(fold_maes, ddof=1)), "cv_seconds": time.perf_counter() - start}

def tail_report(model_name, y_true, y_pred, p90, p99):
    rows = []
    for label, mask in [("loss < train p90", y_true < p90), ("train p90 <= loss < p99", (y_true >= p90) & (y_true < p99)),
                        ("loss >= train p99", y_true >= p99), ("all validation claims", y_true == y_true)]:
        rows.append({"model": model_name, "partition": "validation", "segment": label,
                     "support_claims": int(mask.sum()), "mae_original_loss_units": mae(y_true[mask], y_pred[mask])})
    return rows

P90, P99 = train["loss"].quantile(0.90), train["loss"].quantile(0.99)
print(f"train p90 = {P90:.2f}, p99 = {P99:.2f}")